# Demo: Part 3 - multi-agent image assistant (Gradio)
Planner → Vision → Prompt Engineering → Image Generation → Critique (at most 2 revisions), from tag `p3-v1.0`.

**Before the demo:** Colab, Runtime → **T4 GPU**. Secrets: `GITHUB_TOKEN`, `ANTHROPIC_API_KEY`. Run all cells
once: the warm-up downloads SD 1.5, instruct-pix2pix, the LoRA and CLIP (about 5 min) so the live requests are fast.
Test images are in `part3_agents/test_images/` (also on your laptop in the repo).

In [ ]:
# Cell 1 - run information: clone the graded tag, then print commit, versions, GPU and seed
import os, sys, subprocess, platform, random, datetime
import numpy as np
TAG, REPO, WORK = "p3-v1.0", "sakotiya/data266-lab2-team32", "/content/data266-lab2-team32"
from google.colab import userdata
if not os.path.isdir(WORK):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "-q", "-b", TAG, f"https://{token}@github.com/{REPO}.git", WORK], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)
SEED = 266; random.seed(SEED); np.random.seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print({"part": 3, "tag": TAG, "commit": commit, "python": platform.python_version(),
       "gpu": gpu or "cpu", "seed": SEED, "time": datetime.datetime.now().isoformat(timespec="minutes")})

In [ ]:
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao     # Colab's torchao 0.10 makes peft refuse to load the LoRA
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
# The adapter Part 3 was evaluated with (rank 32). For Part 2's final model use ...-run1-lora-r4.
os.environ["P3_LORA"] = "sakotiya/data266-team32-naruto-run2-lora-r32"
import anthropic, diffusers, transformers, gradio
from part3_agents.src import models
print({"diffusers": diffusers.__version__, "transformers": transformers.__version__, "gradio": gradio.__version__,
       "anthropic": anthropic.__version__, "vlm": models.VLM, "lora": models.LORA, "device": models.DEVICE})

## Warm-up: load every model once

In [ ]:
import time
t = time.perf_counter()
models.sd_img2img(); models.load_naruto_lora(); models.pix2pix(); models.clip()
print(f"all models loaded in {time.perf_counter() - t:.0f} s")

## Gradio app (share link opens in a new tab)

In [ ]:
from part3_agents.app.app import demo
demo.launch(share=True, debug=False)

## Backup: run a scenario without the UI
If the Gradio link fails, run any scenario or red-team case by ID and show every agent's output and the trace.
Good picks: `S01` (caption), `S07` (edit), `S10` (Naruto style), **`S15` or `R09` (broken file, fails safely)**,
`R02` (hidden instruction in the image), `R04` (unsafe request, refused).

In [ ]:
import json, pandas as pd
from IPython.display import display
from part3_agents.src.pipeline import run
cases = pd.concat([pd.read_csv("part3_agents/eval/scenarios.csv").rename(columns={"scenario_id": "id"}),
                   pd.read_csv("part3_agents/eval/redteam.csv").rename(columns={"case_id": "id"})])

def demo_case(case_id, use_critic=False):
    c = cases.set_index("id").loc[case_id]
    print(f"{case_id}: {c['request']}  (image: {c['image']})")
    s = run(f"part3_agents/test_images/{c['image']}", c["request"], use_critic=use_critic)
    print("route:", s.route, "|", s.message, f"| {s.seconds}s | trace: {s.trace_path}")
    for step in s.trace:
        print(f"--- {step['agent']} ({step['seconds']}s)")
        print(json.dumps(step.get("output", step.get("error")), indent=1, default=str)[:700])
    if s.final_image is not None:
        display(s.final_image)
    return s

_ = demo_case("S15")

In [ ]:
_ = demo_case("S07")